# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [ ]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 2000
SPATIAL_UNIT = "community_area" # options: census_tract, h3_cell, community_area
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4h" # options: 1h, 2h, 4h

In [ ]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder
import pytorch_lightning as py_light

from sklearn.preprocessing import OneHotEncoder

## Preparations

In [ ]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [ ]:
# "Settings" / Decisions for the training data

DATA_PATH_TRAIN = INPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_train.parquet"
DATA_PATH_TEST = INPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_test.parquet"


MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
]

        




Load data and select features and target

In [ ]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
test = pl.scan_parquet(DATA_PATH_TEST)

In [ ]:
keep_cols = [
    SPATIAL_UNIT,  # "h3_cell"
    "trip_count",  
    "month_sin", "month_cos", "weekday_sin", "weekday_cos",
    "hour_sin", "hour_cos",
    "tmpc", "relh", "sknt", "vsby", "p01m",
    "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
    "is_holiday",
    "food_drink", "landmark", "shop", "train_station",
]

train_df = train.select(keep_cols).collect().to_pandas()
test_df = test.select(keep_cols).collect().to_pandas()

In [6]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

In [7]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2024-05-13 07:00:00,5,1,7,0.866025,-0.500000,0.000000,1.000000,0.965926,-2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2024-05-06 02:00:00,5,1,2,0.866025,-0.500000,0.000000,1.000000,0.500000,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2024-05-25 05:00:00,5,6,5,0.866025,-0.500000,-0.974928,-0.222521,0.965926,2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2024-06-02 02:00:00,6,7,2,0.500000,-0.866025,-0.781831,0.623490,0.500000,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2024-06-02 18:00:00,6,7,18,0.500000,-0.866025,-0.781831,0.623490,-1.000000,-1.836970e-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [10]:
train_df = train_df.sample(n=2000, random_state=42)
train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)

In [ ]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

Spatial Encoding: LatLong

In [ ]:

if (SPATIAL_ENCODING == "latlong"):
    if SPATIAL_UNIT == "h3_cell":
        print("Encoding: latlong and Unit: hexa")
        for df in (train_df, test_df):
            df["lat"], df["lon"] = zip(*df[SPATIAL_UNIT].map(h3.cell_to_latlng))

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        # create 
        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]
    elif SPATIAL_UNIT == "census_tract":
        print("Encoding: latlong and Unit: census_tract")
        # 1. Load the census tract CSV and parse the_geom (WKT) into geometry

        census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
        census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

        tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
        tract_centroids.columns = ["lat", "lon"]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(11)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check — catch silent join failures early
            n_missing = df["lat"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
            n_missing = df["lon"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]
    elif SPATIAL_UNIT == "community_area":
        print("Encoding: latlong and Unit: community_area")
        # 1. Load the community area CSV and parse the_geom (WKT) into geometry
        census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
        census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

        census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  # the_geom is already lon/lat degrees

        gdf["lon"] = gdf.geometry.centroid.x
        gdf["lat"] = gdf.geometry.centroid.y

        tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(2)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check — catch silent join failures early
            n_missing_lat = df["lat"].isna().sum()
            n_missing_lon = df["lon"].isna().sum()
            if n_missing_lat or n_missing_lon:
                print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]

Features: ['month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'is_holiday', 'census_tract', 'food_drink', 'landmark', 'shop', 'train_station']
Target: uint32


Spatial Encoding: Onehot

In [ ]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "community_area"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train (in case a community_area is missing)
    X_test = X_test.reindex(columns=train_columns, fill_value=0)

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
    X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)


Create y

In [ ]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_train_grid = train_df_grid[TARGET_COL]

In [13]:
model = SVR()

In [ ]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.6700166221084465 best params: {'C': 0.1, 'epsilon': 0.1, 'kernel': 'linear'}


In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'degree': 3, 'epsilon': 1, 'gamma': 0.1, 'kernel': 'poly'}
Best CV score: 0.7982840096939031


In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

: 

: 

In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([-11.77651581, -11.93368319,  -7.16353822, ...,   3.56672217,
         2.63542422, -26.8659655 ], shape=(237252,))

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 12.873083194922646
MSE: 1305.446090023576
RMSE: 36.130957502169466
R2 Score: -0.07329398996990988


In [ ]:
# save model
dump(best_model, "../models/test/model_" + SPATIAL_UNIT + "_svr.joblib")
dump(grid_search, "../models/test/grid_" + SPATIAL_UNIT + "_svr.joblib")

['../models/grid_community_svr.joblib']